In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install -q sentence-transformers datasets

In [3]:
!pip install faiss-gpu-cu12


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 38.6 MB/s eta 0:00:00


In [4]:
import faiss
import torch

# Force everything off GPU
FAISS_GPU = False
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f}GB")

VRAM free: 15.5GB


In [5]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║         QRAG NeurIPS Workshop Benchmark — Kaggle GPU Edition            ║
# ║  Run on Kaggle with GPU T4 x2 accelerator enabled                       ║
# ║                                                                          ║
# ║  Setup: In your Kaggle notebook, run this first cell:                   ║
# ║    !pip install -q sentence-transformers datasets
# Note: faiss is pre-installed on Kaggle GPU instances              ║
# ║                                                                          ║
# ║  Then paste this entire file into the next cell and run.                ║
# ║                                                                          ║
# ║  Runs BOTH scifact (5k) and fiqa (57k) in one shot.                     ║
# ║  Saves all figures + CSVs to /kaggle/working/results/                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, time, json, csv, warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# ── Env vars (safe on all platforms) ─────────────────────────────────────────
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# ── Detect GPU ───────────────────────────────────────────────────────────────
import torch
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
GPU_COUNT  = torch.cuda.device_count()
print(f"[HW] device={DEVICE}  GPUs={GPU_COUNT}")
if DEVICE == "cuda":
    print(f"     {torch.cuda.get_device_name(0)}")

# ── FAISS — prefer GPU build on Kaggle ───────────────────────────────────────
try:
    import faiss
    FAISS_OK = True
    # Move FAISS ops to GPU if available
    if DEVICE == "cuda":
        RES        = faiss.StandardGpuResources()
        FAISS_GPU  = True
        print("[OK] faiss-gpu available")
    else:
        FAISS_GPU  = False
        print("[OK] faiss-cpu available")
except ImportError:
    FAISS_OK  = False
    FAISS_GPU = False
    print("[WARN] FAISS not found — on Kaggle it should be pre-installed")

# ── sentence-transformers ─────────────────────────────────────────────────────
try:
    from sentence_transformers import SentenceTransformer
    ST_OK = True
    print("[OK] sentence-transformers available")
except ImportError:
    ST_OK = False
    print("[WARN] sentence-transformers not found")

# ── HuggingFace datasets ──────────────────────────────────────────────────────
try:
    from datasets import load_dataset
    DS_OK = True
    print("[OK] datasets available")
except ImportError:
    DS_OK = False
    print("[WARN] datasets not found")

# ── Output dir ────────────────────────────────────────────────────────────────
OUT_DIR = "/kaggle/working/results"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs("/kaggle/working/data/embeddings", exist_ok=True)


# ─────────────────────────────────────────────────────────────────────────────
# 1.  DATA & ENCODING
# ─────────────────────────────────────────────────────────────────────────────

BEIR_CONFIGS = {
    "scifact" : ("scifact", ("title", "text")),
    "fiqa"    : ("fiqa",    ("text",)),
    "arguana" : ("arguana", ("title", "text")),
    "scidocs" : ("scidocs", ("title", "text")),
}

# Encoding model — loaded once, shared across datasets
_MODEL = None

def get_model():
    global _MODEL
    if _MODEL is None:
        print(f"Loading all-mpnet-base-v2 on {DEVICE} …")
        _MODEL = SentenceTransformer("all-mpnet-base-v2", device=DEVICE)
    return _MODEL


def load_and_encode(dataset_name: str, max_docs: int):
    cache = f"/kaggle/working/data/embeddings/{dataset_name}_{max_docs}.npz"

    if os.path.exists(cache):
        print(f"  Loading cached embeddings: {cache}")
        data = np.load(cache, allow_pickle=True)
        return data["embeddings"].astype("float32"), list(data["texts"])

    mteb_name, fields = BEIR_CONFIGS[dataset_name]
    print(f"  Downloading mteb/{mteb_name} …")
    ds = load_dataset(f"mteb/{mteb_name}", "corpus", split="corpus")

    texts = []
    for row in ds:
        parts = [str(row.get(f, "") or "") for f in fields]
        texts.append(" ".join(p for p in parts if p).strip())
    texts = [t for t in texts if t][:max_docs]
    print(f"  {len(texts):,} documents")

    # GPU batch size: 256 on T4, 512 on A100 — auto-scales
    batch_size = 256 if DEVICE == "cuda" else 32
    print(f"  Encoding on {DEVICE} (batch_size={batch_size}) …")
    model = get_model()
    embs  = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")
    print(f"  Encoded: {embs.shape}")

    np.savez_compressed(cache, embeddings=embs, texts=np.array(texts, dtype=object))
    return embs, texts


# ─────────────────────────────────────────────────────────────────────────────
# 2.  QUANTISERS
# ─────────────────────────────────────────────────────────────────────────────

class FourBitQuantizer:
    levels = 15
    def quantize(self, data):
        mins = np.percentile(data, 1, axis=0)
        maxs = np.percentile(data, 99, axis=0)
        q = np.round(np.clip((data - mins) / np.maximum(maxs - mins, 1e-8), 0, 1)
                     * self.levels).astype(np.uint8)
        N, D = q.shape
        packed = np.zeros((N, (D + 1) // 2), dtype=np.uint8)
        for d in range(D):
            b = d // 2
            if d % 2 == 0: packed[:, b] |= q[:, d]
            else:           packed[:, b] |= (q[:, d] << 4)
        return packed, mins, maxs

    def dequantize(self, packed, mins, maxs, shape):
        N, D = shape
        out = np.zeros((N, D), dtype=np.uint8)
        for d in range(D):
            b = d // 2
            if b < packed.shape[1]:
                if d % 2 == 0: out[:, d] = packed[:, b] & 0x0F
                else:           out[:, d] = (packed[:, b] >> 4) & 0x0F
        return mins + (out.astype("float32") / self.levels) * (maxs - mins)


class EightBitQuantizer:
    levels = 255
    def quantize(self, data):
        mins = np.percentile(data, 1, axis=0)
        maxs = np.percentile(data, 99, axis=0)
        q = np.round(np.clip((data - mins) / np.maximum(maxs - mins, 1e-8), 0, 1)
                     * self.levels).astype(np.uint8)
        return q, mins, maxs

    def dequantize(self, q, mins, maxs, shape):
        return mins + (q.astype("float32") / self.levels) * (maxs - mins)


# ─────────────────────────────────────────────────────────────────────────────
# 3.  QRAG
# ─────────────────────────────────────────────────────────────────────────────

class QRAG:
    def __init__(self, projection_dim: int = 128):
        self.projection_dim = projection_dim

    def fit(self, embeddings: np.ndarray):
        N, D = embeddings.shape
        self.N, self.D = N, D
        pd = min(self.projection_dim, D, N - 1)

        self.pca = PCA(n_components=pd, random_state=0)
        self.pca.fit(embeddings)
        var = float(np.sum(self.pca.explained_variance_ratio_))

        e1_raw  = self.pca.transform(embeddings).astype("float32")
        e1_norm = e1_raw / np.maximum(np.linalg.norm(e1_raw, axis=1, keepdims=True), 1e-12)

        if FAISS_OK:
            cpu_idx = faiss.IndexHNSWFlat(pd, 32)
            cpu_idx.hnsw.efConstruction = 200
            cpu_idx.add(e1_norm)
            # Move to GPU if availabl
            self.index = cpu_idx
        else:
            self.index = e1_norm

        base      = self.pca.inverse_transform(e1_raw).astype("float32")
        residuals = embeddings - base

        self.q4 = FourBitQuantizer()
        self.q8 = EightBitQuantizer()
        self.packed_res, self.res_min, self.res_max = self.q4.quantize(residuals)
        self.packed_e1,  self.e1_min,  self.e1_max  = self.q8.quantize(e1_raw)
        return var

    def search(self, query: np.ndarray, k1: int = 200, k_final: int = 10):
        q_raw  = self.pca.transform(query.reshape(1, -1)).astype("float32")
        q_norm = q_raw / np.maximum(np.linalg.norm(q_raw), 1e-12)

        if FAISS_OK:
            _, ids = self.index.search(q_norm, k1)
            cands  = ids[0]
        else:
            cands = np.argsort(-(self.index @ q_norm.reshape(-1)))[:k1]

        e1_c   = self.q8.dequantize(self.packed_e1[cands], self.e1_min, self.e1_max,
                                    (len(cands), self.pca.n_components_))
        base_c = self.pca.inverse_transform(e1_c).astype("float32")
        res_c  = self.q4.dequantize(self.packed_res[cands], self.res_min, self.res_max,
                                    (len(cands), self.D))
        recon  = base_c + res_c
        recon /= np.maximum(np.linalg.norm(recon, axis=1, keepdims=True), 1e-12)

        qn  = query / np.maximum(np.linalg.norm(query), 1e-12)
        sc  = recon @ qn
        top = np.argsort(-sc)[:k_final]
        return cands[top].tolist(), sc[top]

    def memory_mb(self):
        mb = (self.packed_res.nbytes + self.packed_e1.nbytes +
              self.res_min.nbytes   + self.res_max.nbytes +
              self.e1_min.nbytes    + self.e1_max.nbytes) / 1e6
        # HNSW index memory (CPU estimate even if on GPU)
        mb += self.N * self.pca.n_components_ * 4 / 1e6
        return mb


# ─────────────────────────────────────────────────────────────────────────────
# 4.  FAISS BASELINES
# ─────────────────────────────────────────────────────────────────────────────

def _to_gpu(idx):
    """Move a CPU FAISS index to GPU if available."""
    if FAISS_GPU:
        return faiss.index_cpu_to_gpu(RES, 0, idx)
    return idx

def build_flat(X):
    idx = faiss.IndexFlatIP(X.shape[1])
    idx.add(X)
    return _to_gpu(idx)

def build_ivfflat(X):
    d, n  = X.shape[1], len(X)
    nlist = max(4, int(4 * np.sqrt(n)))
    idx   = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, nlist, faiss.METRIC_INNER_PRODUCT)
    idx.train(X); idx.add(X)
    idx.nprobe = max(1, nlist // 8)
    return _to_gpu(idx)

def build_ivfpq(X, m=None):
    d, n  = X.shape[1], len(X)
    nlist = max(4, int(4 * np.sqrt(n)))
    m     = m or max(4, d // 8)
    while d % m != 0 and m > 1: m -= 1
    idx   = faiss.IndexIVFPQ(faiss.IndexFlatIP(d), d, nlist, m, 8)
    idx.train(X); idx.add(X)
    idx.nprobe = max(1, nlist // 8)
    # IVF-PQ stays on CPU — GPU shared memory limit (49152 bytes) is too small
    # for 96 sub-quantizers (needs 98304 bytes). This is a T4 hardware constraint.
    return idx

def build_hnsw(X):
    # HNSW can't move to GPU — stays on CPU (FAISS limitation)
    idx = faiss.IndexHNSWFlat(X.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
    idx.hnsw.efConstruction = 200
    idx.add(X)
    return idx

def index_memory_mb(name, X, idx):
    n, d = X.shape
    if   name == "IVF-PQ":   return (n * idx.pq.M + idx.nlist * d * 4) / 1e6
    elif name == "IVF-Flat": return (n * d * 4 + idx.nlist * d * 4) / 1e6
    elif name == "HNSW":     return (n * d * 4 + n * 32 * 2 * 4) / 1e6
    else:                    return  n * d * 4 / 1e6

def eval_index(idx, queries, ground_truth, k=10):
    recalls, times = [], []
    for i, q in enumerate(queries):
        t0 = time.time()
        _, ids = idx.search(q.reshape(1, -1).astype("float32"), k)
        times.append(time.time() - t0)
        recalls.append(len(set(ids[0].tolist()) & set(ground_truth[i])) / k)
    return float(np.mean(recalls)), float(np.std(recalls)), float(np.mean(times) * 1000)


# ─────────────────────────────────────────────────────────────────────────────
# 5.  SWEEP
# ─────────────────────────────────────────────────────────────────────────────

def qrag_sweep(embeddings, queries, ground_truth, projection_dims, k1=200, k=10):
    results = []
    for pd in projection_dims:
        print(f"  QRAG dim={pd:>4d} … ", end="", flush=True)
        q   = QRAG(projection_dim=pd)
        var = q.fit(embeddings)
        recalls, times = [], []
        for i, qv in enumerate(queries):
            t0 = time.time()
            ids, _ = q.search(qv, k1=k1, k_final=k)
            times.append(time.time() - t0)
            recalls.append(len(set(ids) & set(ground_truth[i])) / k)
        mem = q.memory_mb()
        r   = float(np.mean(recalls))
        print(f"recall={r:.4f}  mem={mem:.1f}MB  var={var:.3f}")
        results.append({"label": f"QRAG-{pd}", "projection_dim": pd,
                        "recall": r, "recall_std": float(np.std(recalls)),
                        "memory_mb": mem, "latency_ms": float(np.mean(times)*1000),
                        "pca_variance": var})
    return results


# ─────────────────────────────────────────────────────────────────────────────
# 6.  PLOTS
# ─────────────────────────────────────────────────────────────────────────────

COLOURS = {"Flat":"#dc2626","IVF-Flat":"#d97706","IVF-PQ":"#16a34a",
           "HNSW":"#7c3aed","QRAG":"#2563eb"}
MARKERS = {"Flat":"s","IVF-Flat":"^","IVF-PQ":"D","HNSW":"P"}

def plot_pareto(qrag_pts, baselines, base_mem, fp32_mb, dataset, out):
    fig, ax = plt.subplots(figsize=(10, 6.5))
    ax.set_facecolor("#f8f9fa")

    order = np.argsort([p["memory_mb"] for p in qrag_pts])
    ax.plot([qrag_pts[i]["memory_mb"] for i in order],
            [qrag_pts[i]["recall"]    for i in order],
            "o-", color=COLOURS["QRAG"], lw=2.5, ms=8, label="QRAG (this work)", zorder=6)
    for p in qrag_pts:
        ax.annotate(f"dim={p['projection_dim']}", (p["memory_mb"], p["recall"]),
                    textcoords="offset points", xytext=(6, 4), fontsize=7.5, color="#1e40af")

    for name, rec in baselines.items():
        mem = base_mem[name]
        ax.scatter(mem, rec, s=160, color=COLOURS.get(name,"grey"),
                   marker=MARKERS.get(name,"o"), zorder=7,
                   label=f"{name}  R={rec:.3f}  {mem:.0f}MB")
        ax.annotate(name, (mem, rec), textcoords="offset points",
                    xytext=(6,-12), fontsize=8.5, color=COLOURS.get(name,"grey"), fontweight="bold")

    ax.axvline(fp32_mb, color="#dc2626", ls="--", lw=1.2, alpha=0.4,
               label=f"FP32 ({fp32_mb:.0f}MB)")
    ax.set_xlabel("Memory (MB)", fontsize=13)
    ax.set_ylabel("Recall@10", fontsize=13)
    ax.set_title(f"Recall@10 vs Memory — {dataset.upper()} (768-dim)\nUpper-left = better",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=8.5, loc="lower right", framealpha=0.9)
    ax.set_ylim(0, 1.06); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(out, dpi=160); plt.close()
    print(f"  Saved: {out}")


def plot_compression_recall(qrag_pts, baselines, base_mem, fp32_mb, dataset, out):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_facecolor("#f8f9fa")

    order = np.argsort([p["memory_mb"] for p in qrag_pts])
    ax.plot([fp32_mb / qrag_pts[i]["memory_mb"] for i in order],
            [qrag_pts[i]["recall"]              for i in order],
            "o-", color=COLOURS["QRAG"], lw=2.5, ms=8, label="QRAG (this work)", zorder=6)
    for p in qrag_pts:
        ax.annotate(f"dim={p['projection_dim']}",
                    (fp32_mb / p["memory_mb"], p["recall"]),
                    textcoords="offset points", xytext=(4, 5), fontsize=7.5, color="#1e40af")

    for name, rec in baselines.items():
        cx = fp32_mb / base_mem[name]
        ax.scatter(cx, rec, s=160, color=COLOURS.get(name,"grey"),
                   marker=MARKERS.get(name,"o"), zorder=7,
                   label=f"{name}  {cx:.1f}x  R={rec:.3f}")
        ax.annotate(name, (cx, rec), textcoords="offset points",
                    xytext=(4,-13), fontsize=8.5, color=COLOURS.get(name,"grey"), fontweight="bold")

    ax.set_xlabel("Compression (× FP32)", fontsize=13)
    ax.set_ylabel("Recall@10", fontsize=13)
    ax.set_title(f"Compression vs Recall — {dataset.upper()} (768-dim)\nUpper-right = better",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=8.5, loc="lower left", framealpha=0.9)
    ax.set_ylim(0, 1.06); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(out, dpi=160); plt.close()
    print(f"  Saved: {out}")


def plot_pca_variance(qrag_pts, dataset, out):
    dims  = [p["projection_dim"] for p in qrag_pts]
    vars_ = [p["pca_variance"]   for p in qrag_pts]
    recs  = [p["recall"]         for p in qrag_pts]

    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax2 = ax1.twinx(); ax1.set_facecolor("#f8f9fa")
    ax1.plot(dims, vars_, "s--", color="#d97706", lw=2, ms=7, label="PCA variance")
    ax2.plot(dims, recs,  "o-",  color="#2563eb", lw=2, ms=7, label="Recall@10")
    ax1.set_xlabel("Projection Dimension", fontsize=12)
    ax1.set_ylabel("PCA Variance Explained", fontsize=12, color="#d97706")
    ax2.set_ylabel("Recall@10", fontsize=12, color="#2563eb")
    ax1.set_title(f"PCA Variance & Recall vs Projection Dim — {dataset.upper()}", fontsize=12, fontweight="bold")
    lines  = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
    labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
    ax1.legend(lines, labels, fontsize=9)
    ax1.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(out, dpi=160); plt.close()
    print(f"  Saved: {out}")


def plot_combined_pareto(all_results: dict, out):
    """One plot with both datasets — the paper's main figure."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
    ds_styles = {"scifact": ("o-", "#2563eb"), "fiqa": ("s--", "#7c3aed")}

    for ax, (ds_name, ds_data) in zip(axes, all_results.items()):
        qrag_pts  = ds_data["qrag_pts"]
        baselines = ds_data["baselines"]
        base_mem  = ds_data["base_mem"]
        fp32_mb   = ds_data["fp32_mb"]

        order = np.argsort([p["memory_mb"] for p in qrag_pts])
        marker, colour = ds_styles[ds_name]
        ax.plot([qrag_pts[i]["memory_mb"] for i in order],
                [qrag_pts[i]["recall"]    for i in order],
                marker, color=colour, lw=2.5, ms=8, label="QRAG (this work)")

        for name, rec in baselines.items():
            mem = base_mem[name]
            ax.scatter(mem, rec, s=140, color=COLOURS.get(name,"grey"),
                       marker=MARKERS.get(name,"o"),
                       label=f"{name} R={rec:.3f}")
            ax.annotate(name, (mem, rec), textcoords="offset points",
                        xytext=(5,-12), fontsize=8, color=COLOURS.get(name,"grey"), fontweight="bold")

        ax.axvline(fp32_mb, color="#dc2626", ls="--", lw=1, alpha=0.4)
        ax.set_title(f"{ds_name.upper()} — {ds_data['n_docs']:,} docs", fontsize=13, fontweight="bold")
        ax.set_xlabel("Memory (MB)", fontsize=12)
        ax.set_ylabel("Recall@10", fontsize=12)
        ax.set_ylim(0, 1.06); ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8, loc="lower right", framealpha=0.9)
        ax.set_facecolor("#f8f9fa")

    plt.suptitle("QRAG vs FAISS Baselines — 768-dim BEIR Benchmarks",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(out, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {out}")


# ─────────────────────────────────────────────────────────────────────────────
# 7.  RUN ONE DATASET
# ─────────────────────────────────────────────────────────────────────────────

def run_dataset(dataset_name: str, max_docs: int, n_queries: int,
                projection_dims: list, k: int = 10, k1: int = 200) -> dict:

    print(f"\n{'='*70}")
    print(f"  DATASET: {dataset_name.upper()}  ({max_docs:,} docs)")
    print(f"{'='*70}")

    embeddings, texts = load_and_encode(dataset_name, max_docs)
    N, D = embeddings.shape
    fp32_mb = N * D * 4 / 1e6
    print(f"\n  {N:,} docs × dim {D}  ({fp32_mb:.1f} MB FP32)")

    # Queries + ground truth
    rng   = np.random.RandomState(42)
    n_q   = min(n_queries, N)
    q_idx = rng.choice(N, n_q, replace=False)
    queries = []
    for i in q_idx:
        q  = embeddings[i] + 0.02 * rng.normal(size=D).astype("float32")
        q /= np.maximum(np.linalg.norm(q), 1e-12)
        queries.append(q)

    print(f"\n  Computing exact ground truth for {n_q} queries …")
    ground_truth = [np.argsort(-(embeddings @ q))[:k].tolist() for q in queries]

    # FAISS baselines
    baseline_results, baseline_mem = {}, {}
    if FAISS_OK:
        configs = [
            ("Flat",     lambda: build_flat(embeddings)),
            ("IVF-Flat", lambda: build_ivfflat(embeddings)),
            ("IVF-PQ",   lambda: build_ivfpq(embeddings)),
            ("HNSW",     lambda: build_hnsw(embeddings)),
        ]
        for name, builder in configs:
            print(f"  Building {name} … ", end="", flush=True)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")   # suppress nlist warnings
                t0  = time.time()
                idx = builder()
                bt  = time.time() - t0
            r, std, lat = eval_index(idx, queries, ground_truth, k)
            # For GPU indexes we need the CPU version to get nlist/pq attrs
            cpu_idx = faiss.index_gpu_to_cpu(idx) if FAISS_GPU and name != "HNSW" else idx
            mem = index_memory_mb(name, embeddings, cpu_idx)
            baseline_results[name] = r
            baseline_mem[name]     = mem
            print(f"recall={r:.4f} ±{std:.4f}  mem={mem:.1f}MB  build={bt:.1f}s  lat={lat:.2f}ms")

    # QRAG sweep
    pds = [p for p in projection_dims if p < D]
    print(f"\n  QRAG sweep over dims: {pds}")
    qrag_pts = qrag_sweep(embeddings, queries, ground_truth, pds, k1=k1, k=k)

    # Per-dataset plots
    tag = f"{dataset_name}_{max_docs}"
    plot_pareto(qrag_pts, baseline_results, baseline_mem, fp32_mb, dataset_name,
                f"{OUT_DIR}/pareto_{tag}.png")
    plot_compression_recall(qrag_pts, baseline_results, baseline_mem, fp32_mb, dataset_name,
                            f"{OUT_DIR}/compression_{tag}.png")
    plot_pca_variance(qrag_pts, dataset_name,
                      f"{OUT_DIR}/pca_variance_{tag}.png")

    # CSV
    rows = []
    for name, rec in baseline_results.items():
        rows.append({"method": name, "recall@10": f"{rec:.4f}",
                     "memory_mb": f"{baseline_mem[name]:.1f}",
                     "compression": f"{fp32_mb/baseline_mem[name]:.2f}x",
                     "pca_variance": "—"})
    for p in qrag_pts:
        rows.append({"method": p["label"], "recall@10": f"{p['recall']:.4f}",
                     "memory_mb": f"{p['memory_mb']:.1f}",
                     "compression": f"{fp32_mb/p['memory_mb']:.2f}x",
                     "pca_variance": f"{p['pca_variance']:.3f}"})
    with open(f"{OUT_DIR}/table_{tag}.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)

    # Print table
    print(f"\n  {'Method':<18} {'Recall@10':>10} {'Memory(MB)':>12} {'Compression':>13} {'PCA Var':>9}")
    print(f"  {'-'*65}")
    for r in rows:
        print(f"  {r['method']:<18} {r['recall@10']:>10} {r['memory_mb']:>12} "
              f"{r['compression']:>13} {r['pca_variance']:>9}")

    return {"qrag_pts": qrag_pts, "baselines": baseline_results,
            "base_mem": baseline_mem, "fp32_mb": fp32_mb, "n_docs": N,
            "rows": rows}


# ─────────────────────────────────────────────────────────────────────────────
# 8.  MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    PROJECTION_DIMS = [32, 64, 96, 128, 192, 256, 384]

    all_results = {}

    # ── Dataset 2: fiqa (57k, the real test) ─────────────────────────────────
    all_results["fiqa"] = run_dataset(
        dataset_name    = "fiqa",
        max_docs        = 57_000,    # full corpus — GPU makes this fast
        n_queries       = 500,
        projection_dims = PROJECTION_DIMS,
    )

    # ── Combined figure (the paper's main figure) ─────────────────────────────
    print("\nGenerating combined figure …")
    plot_combined_pareto(all_results, f"{OUT_DIR}/combined_pareto.png")

    # ── Combined JSON ─────────────────────────────────────────────────────────
    with open(f"{OUT_DIR}/all_results.json", "w") as f:
        json.dump({ds: {k: v for k, v in data.items() if k != "rows"}
                   for ds, data in all_results.items()}, f, indent=2, default=str)

    print(f"\n{'='*70}")
    print(f"  All outputs saved to {OUT_DIR}/")
    print(f"  Key figure for paper: {OUT_DIR}/combined_pareto.png")
    print(f"{'='*70}")

[HW] device=cuda  GPUs=2
     Tesla T4
[OK] faiss-gpu available


[OK] sentence-transformers available
[OK] datasets available

  DATASET: FIQA  (57,000 docs)


README.md: 0.00B [00:00, ?B/s]

corpus.jsonl:   0%|          | 0.00/47.0M [00:00<?, ?B/s]

Generating corpus split:   0%|          | 0/57638 [00:00<?, ? examples/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/223 [00:00<?, ?it/s]

  Encoded: (57000, 768)

  57,000 docs × dim 768  (175.1 MB FP32)

  Computing exact ground truth for 500 queries …
  Building Flat … recall=1.0000 ±0.0000  mem=175.1MB  build=0.3s  lat=1.10ms
  Building IVF-Flat … recall=0.9980 ±0.0166  mem=178.0MB  build=13.4s  lat=0.42ms
  Building IVF-PQ … recall=0.7356 ±0.1264  mem=8.4MB  build=138.2s  lat=22.57ms
  Building HNSW … recall=0.9924 ±0.0287  mem=189.7MB  build=23.8s  lat=0.19ms

  QRAG sweep over dims: [32, 64, 96, 128, 192, 256, 384]
  QRAG dim=  32 … recall=0.8596  mem=31.0MB  var=0.441
  QRAG dim=  64 … recall=0.9160  mem=40.1MB  var=0.594
  QRAG dim=  96 … recall=0.9428  mem=49.3MB  var=0.689
  QRAG dim= 128 … recall=0.9514  mem=58.4MB  var=0.759
  QRAG dim= 192 … recall=0.9500  mem=76.6MB  var=0.854
  QRAG dim= 256 … recall=0.9538  mem=94.9MB  var=0.912
  QRAG dim= 384 … recall=0.9548  mem=131.3MB  var=0.965
  Saved: /kaggle/working/results/pareto_fiqa_57000.png
  Saved: /kaggle/working/results/compression_fiqa_57000.png
  Saved: